# NeuroGolf 2026 — per-task MAX BLEND of two submissions

You have two submissions:
- **A** = your current best bundle (LB 6202.05) — a DIR of task*.onnx
- **B** = the 6151 FP16-surgery bundle — its submission.zip

Neither dominates the other — each wins some tasks. This notebook builds a
**per-task maximum**: for every task001..task400 it scores *both* candidates
through the official scorer and keeps the one that is **correct AND cheaper**
(higher points). The blended zip therefore scores **>= max(A, B)** on every task,
so the blended LB is at least as high as the better of the two, and usually higher.

### Safety
This is a *selection* blend, not a learned rebuild. It only ever picks an ONNX
that one of your real submissions already contains — so whatever each file scored
on the private set, it still scores. There is **no new model and no private-set
risk**: the blend cannot do worse than the better source on any task.

### Tie-breaking rules (per task)
1. If only one candidate is correct (n_fail==0) -> pick it.
2. If both correct -> pick lower cost (higher points).
3. If both correct and equal cost -> pick A (arbitrary, stable).
4. If neither correct locally -> pick the one with fewer fails; if still tied,
   pick A. (Shouldn't happen for real submissions, but handled.)

### Attach in the right panel
- Competition task JSONs: `/kaggle/input/competitions/neurogolf-2026`
- Submission **A** ONNX: a dataset dir OR its `submission.zip` (set A_DIR or A_ZIP)
- Submission **B** ONNX: a dataset dir OR its `submission.zip` (set B_DIR or B_ZIP)

Set the paths in Cell 2, then run top to bottom. Output: `/kaggle/working/submission.zip`.

## Cell 0 — deps

In [ ]:
import sys, subprocess
for pkg in ["onnxruntime","onnx"]:
    try: subprocess.run([sys.executable,"-m","pip","install","-q",pkg],check=False)
    except Exception as e: print("pip note:",pkg,e)
print("deps done")


## Cell 1 — Audit harness (official-scorer mirror; defines audit_one, SOURCE)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
NeuroGolf 2026 -- Local Audit Harness
=====================================
Re-implements the OFFICIAL scoring pipeline (from neurogolf_utils.py) so you can,
for every task ONNX you already have:

  1. verify functional correctness across ALL valid (<=30x30) examples
     (train + test + arc-gen), exactly as the Kaggle validator does, and
  2. compute the exact cost = params + memory_bytes and the resulting
     points = max(1, 25 - ln(max(1, cost))),

then emit a per-task CSV sorted by points ASCENDING so the tasks bleeding the
most score float straight to the top of your worklist.

Why this matters: score = 25 - ln(cost) is logarithmic, so the entire game is
dragging your WORST tasks down by orders of magnitude. This tells you which
those are, and whether any are silently failing correctness.

USAGE (Kaggle notebook or local):
  # point these at your dataset + the competition task JSONs
  python neurogolf_audit.py \
      --onnx_dir /kaggle/input/datasets/biohack44/neurogolf-6124-bundle \
      --task_dir /kaggle/input/competitions/neurogolf-2026 \
      --out      /kaggle/working/audit.csv

  # or just import and call audit_dir(...) from a notebook cell.

NOTES
- This mirrors neurogolf_utils exactly: sanitize_model -> InferenceSession with
  ORT_DISABLE_ALL + profiling -> calculate_memory(profile trace) + calculate_params.
- It does NOT require onnx_tool (that's only used by the official notebook for a
  cosmetic profile print). Pure onnx + onnxruntime + numpy.
- If neurogolf_utils is importable, we reuse its functions verbatim (safest).
  Otherwise we fall back to a faithful local re-implementation.
"""

import argparse
import csv
import glob
import json
import math
import os
import pathlib
import sys
import traceback

import numpy as np
import onnx
import onnxruntime

# ----------------------------------------------------------------------------
# Try to use the OFFICIAL utils verbatim. This is the safest source of truth.
# IMPORTANT: a module named `neurogolf_utils` may exist on the path that is NOT
# the real competition utils (a stub, an empty namespace pkg, or a partial copy).
# So we do NOT trust a successful import alone -- we verify it actually exposes
# the functions we need. If it doesn't, we fall back to the local re-impl.
# ----------------------------------------------------------------------------
_REQUIRED = ("convert_to_numpy", "sanitize_model", "calculate_params",
             "calculate_memory", "score_network")
_OFFICIAL = None
for cand in [
    "/kaggle/input/competitions/neurogolf-2026",
    "/kaggle/usr/lib/neurogolf_utils",
    "./neurogolf_utils",
    ".",
]:
    if cand not in sys.path:
        sys.path.insert(0, cand)
try:
    import neurogolf_utils as _cand_mod  # type: ignore
    if all(hasattr(_cand_mod, fn) for fn in _REQUIRED):
        _OFFICIAL = _cand_mod
    else:
        _missing = [fn for fn in _REQUIRED if not hasattr(_cand_mod, fn)]
        print(f"[audit] WARNING: found a 'neurogolf_utils' missing {_missing}; "
              f"using local fallback scorer instead.")
        _OFFICIAL = None
except Exception as _e:
    # import itself failed (e.g. missing IPython/matplotlib/onnx_tool) -> fallback
    _OFFICIAL = None

_BATCH, _CH, _H, _W = 1, 10, 30, 30
_GRID_SHAPE = [_BATCH, _CH, _H, _W]


# ----------------------------------------------------------------------------
# Faithful fallback re-implementations (used only if official import fails).
# Kept byte-for-byte consistent with the 2026-05-14 neurogolf_utils.py.
# ----------------------------------------------------------------------------
def _convert_to_numpy(example):
    benchmark = {}
    shape = (1, _CH, _H, _W)
    for mode in ["input", "output"]:
        benchmark[mode] = np.zeros(shape, dtype=np.float32)
        grid = example[mode]
        if max(len(grid), len(grid[0])) > 30:
            return None
        for r, _ in enumerate(grid):
            for c, color in enumerate(grid[r]):
                benchmark[mode][0][color][r][c] = 1.0
    return benchmark


def _sanitize_model(model):
    for node in model.graph.node:
        node.name = node.output[0]
        if "kernel_time" in node.output[0]:
            return None
    name_map, counter = {}, [0]

    def safe(old):
        if not old or old in ["input", "output"]:
            return old
        if old not in name_map:
            name_map[old] = f"safe_name_{counter[0]}"
            counter[0] += 1
        return name_map[old]

    for inp in model.graph.input:
        inp.name = safe(inp.name)
    for init in model.graph.initializer:
        init.name = safe(init.name)
    for node in model.graph.node:
        for i in range(len(node.input)):
            node.input[i] = safe(node.input[i])
        for i in range(len(node.output)):
            node.output[i] = safe(node.output[i])
        if len(node.output) > 0 and node.output[0]:
            node.name = node.output[0]
    for out in model.graph.output:
        out.name = safe(out.name)
    for vi in model.graph.value_info:
        vi.name = safe(vi.name)
    for node in model.graph.node:
        node.name = node.output[0]
    return model


def _calculate_params(model):
    params = 0
    for init in model.graph.initializer:
        if any(d <= 0 for d in init.dims):
            return None
        params += int(np.prod(init.dims))
    for sp in model.graph.sparse_initializer:
        if any(d <= 0 for d in sp.values.dims):
            return None
        params += int(np.prod(sp.values.dims))
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                if any(d <= 0 for d in attr.t.dims):
                    return None
                params += int(np.prod(attr.t.dims))
            elif attr.name == "sparse_value":
                if any(d <= 0 for d in attr.sparse_tensor.values.dims):
                    return None
                params += int(np.prod(attr.sparse_tensor.values.dims))
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return params


_EXCLUDED = ["LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"]


def _calculate_memory(model, trace_path):
    onnx.checker.check_model(model, full_check=True)
    graph = onnx.shape_inference.infer_shapes(model, strict_mode=True).graph
    if len(graph.input) > 1 or len(graph.output) > 1:
        return None
    init_names = {i.name for i in graph.initializer}
    init_names.update(i.name for i in graph.sparse_initializer)
    io_names = {t.name for t in list(graph.input) + list(graph.output)}
    if io_names.intersection(init_names):
        return None
    if model.functions:
        return None
    for opset in model.opset_import:
        if opset.domain not in {"", "ai.onnx"}:
            return None
    node_outputs, tensor_names = {}, set()
    for node in graph.node:
        for attr in node.attribute:
            if attr.type in [onnx.AttributeProto.GRAPH, onnx.AttributeProto.GRAPHS]:
                return None
        node_outputs[node.name] = list(node.output)
        for o in node.output:
            if o:
                tensor_names.add(o)
    tensor_memory, tensor_dtypes = {}, {}
    tensor_map = {t.name: t for t in list(graph.input) + list(graph.value_info) + list(graph.output)}
    tensor_names.update(tensor_map.keys())
    for tn in tensor_names:
        item = tensor_map.get(tn)
        if not item:
            return None
        if item.type.HasField("sequence_type"):
            return None
        if not item.type.HasField("tensor_type"):
            continue
        tt = item.type.tensor_type
        if not tt.HasField("shape"):
            return None
        n = 1
        for dim in tt.shape.dim:
            if dim.HasField("dim_param"):
                return None
            if not dim.HasField("dim_value"):
                return None
            if dim.dim_value <= 0:
                return None
            n *= dim.dim_value
        if tn in ["input", "output"]:
            continue
        npd = onnx.helper.tensor_dtype_to_np_dtype(tt.elem_type)
        tensor_memory[tn] = n * np.dtype(npd).itemsize
        tensor_dtypes[tn] = npd
    seen = set()
    for item in list(graph.input) + list(graph.value_info) + list(graph.output):
        if item.name in seen:
            return None
        seen.add(item.name)
    for node in graph.node:
        for o in node.output:
            if o and o != "output":
                item = tensor_map.get(o)
                if item is None or not item.type.HasField("tensor_type"):
                    return None
    with open(trace_path, "r") as f:
        trace = json.load(f)
    for ev in trace:
        if ev.get("cat") != "Node" or "args" not in ev:
            continue
        if "output_type_shape" not in ev["args"]:
            continue
        nm = ev.get("name").replace("_kernel_time", "")
        if nm not in node_outputs:
            continue
        for i, sd in enumerate(ev["args"]["output_type_shape"]):
            if i >= len(node_outputs[nm]):
                continue
            on = node_outputs[nm][i]
            if on not in tensor_dtypes:
                continue
            isz = np.dtype(tensor_dtypes[on]).itemsize
            mem = isz * sum(int(np.prod(d)) for d in sd.values())
            tensor_memory[on] = max(tensor_memory[on], mem)
    return sum(tensor_memory.values())


# Bind to official versions when available -----------------------------------
if _OFFICIAL is not None:
    convert_to_numpy = _OFFICIAL.convert_to_numpy
    sanitize_model = _OFFICIAL.sanitize_model
    calculate_params = _OFFICIAL.calculate_params
    calculate_memory = _OFFICIAL.calculate_memory
    SOURCE = "official neurogolf_utils"
else:
    convert_to_numpy = _convert_to_numpy
    sanitize_model = _sanitize_model
    calculate_params = _calculate_params
    calculate_memory = _calculate_memory
    SOURCE = "local fallback (official utils not importable)"


# ----------------------------------------------------------------------------
def _load_task(task_dir, task_num):
    p = os.path.join(task_dir, f"task{task_num:03d}.json")
    if not os.path.isfile(p):
        return None
    with open(p) as f:
        return json.load(f)


def _all_valid_examples(examples):
    out = []
    for key in ["train", "test", "arc-gen"]:
        for ex in examples.get(key, []):
            b = convert_to_numpy(ex)
            if b is not None:
                out.append((ex, b))
    return out


def audit_one(onnx_path, examples, run_correctness=True):
    """Returns dict with params, memory, cost, points, n_pass, n_fail, status."""
    res = dict(onnx=os.path.basename(onnx_path), params=None, memory=None,
               cost=None, points=None, n_pass=0, n_fail=0, status="ok",
               filesize=os.path.getsize(onnx_path))

    if res["filesize"] > 1.44 * 1024 * 1024:
        res["status"] = "FILESIZE_OVER_LIMIT"
        res["points"] = 0.0
        return res

    try:
        model = onnx.load(onnx_path)
    except Exception as e:
        res["status"] = f"load_error:{e}"
        res["points"] = 0.0
        return res

    # banned op check (mirror score_network)
    for node in model.graph.node:
        if node.op_type.upper() in _EXCLUDED or "Sequence" in node.op_type:
            res["status"] = f"BANNED_OP:{node.op_type}"
            res["points"] = 0.0
            return res

    # sanitize + session w/ profiling, exactly like the official validator
    try:
        sanitized = sanitize_model(onnx.load(onnx_path))
        if not sanitized:
            res["status"] = "sanitize_failed"
            res["points"] = 0.0
            return res
        opts = onnxruntime.SessionOptions()
        opts.enable_profiling = True
        opts.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
        # unique prefix avoids profile clobbering across tasks
        opts.profile_file_prefix = f"audit_{res['onnx']}"
        sess = onnxruntime.InferenceSession(sanitized.SerializeToString(), opts)
    except Exception as e:
        res["status"] = f"session_error:{e}"
        res["points"] = 0.0
        return res

    # correctness over ALL valid examples (exact array equality, >0 decode)
    if run_correctness and examples is not None:
        valid = _all_valid_examples(examples)
        for ex, b in valid:
            try:
                out = sess.run(["output"], {"input": b["input"]})[0]
                pred = (out > 0.0).astype(np.float32)
                if np.array_equal(pred, b["output"]):
                    res["n_pass"] += 1
                else:
                    res["n_fail"] += 1
            except Exception:
                res["n_fail"] += 1

    trace_path = sess.end_profiling()
    try:
        mem = calculate_memory(sanitized, trace_path)
        params = calculate_params(sanitized)
    except Exception as e:
        res["status"] = f"score_error:{e}"
        res["points"] = 0.0
        try:
            os.remove(trace_path)
        except Exception:
            pass
        return res
    try:
        os.remove(trace_path)
    except Exception:
        pass

    if mem is None or params is None or mem < 0 or params < 0:
        res["status"] = "unscorable"
        res["points"] = 0.0
        return res

    res["params"] = int(params)
    res["memory"] = int(mem)
    res["cost"] = int(params + mem)
    # If any example fails, the network earns ZERO for that task on the real LB.
    if run_correctness and res["n_fail"] > 0:
        res["status"] = "INCORRECT"
        res["points"] = 0.0
    else:
        res["points"] = max(1.0, 25.0 - math.log(max(1.0, params + mem)))
    return res


def audit_dir(onnx_dir, task_dir=None, out_csv=None, run_correctness=True,
              limit=None, verbose=True):
    onnx_files = sorted(glob.glob(os.path.join(onnx_dir, "task*.onnx")))
    if limit:
        onnx_files = onnx_files[:limit]
    if verbose:
        print(f"[audit] scoring source: {SOURCE}")
        print(f"[audit] found {len(onnx_files)} onnx files in {onnx_dir}")
        print(f"[audit] correctness check: {'ON' if run_correctness else 'OFF'}")

    rows = []
    for i, op in enumerate(onnx_files):
        base = os.path.basename(op)
        try:
            tnum = int("".join(ch for ch in base if ch.isdigit())[:3])
        except Exception:
            tnum = None
        examples = _load_task(task_dir, tnum) if (task_dir and tnum) else None
        r = audit_one(op, examples, run_correctness=run_correctness and examples is not None)
        r["task"] = tnum
        rows.append(r)
        if verbose and (i + 1) % 25 == 0:
            print(f"  .. {i+1}/{len(onnx_files)} done")

    # Sort by points ASC (worst first) so the worklist is the top of the file.
    rows.sort(key=lambda x: (x["points"] if x["points"] is not None else -1,
                             x["cost"] if x["cost"] is not None else 1e18))

    if out_csv:
        with open(out_csv, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["task", "onnx", "points", "cost", "params", "memory",
                        "filesize", "n_pass", "n_fail", "status"])
            for r in rows:
                w.writerow([r["task"], r["onnx"],
                            f"{r['points']:.4f}" if r["points"] is not None else "",
                            r["cost"], r["params"], r["memory"], r["filesize"],
                            r["n_pass"], r["n_fail"], r["status"]])
        if verbose:
            print(f"[audit] wrote {out_csv}")

    # Summary
    total = sum(r["points"] for r in rows if r["points"] is not None)
    n_incorrect = sum(1 for r in rows if r["status"] == "INCORRECT")
    n_problem = sum(1 for r in rows if r["status"] not in ("ok", "INCORRECT"))
    missing = 400 - len(rows)
    if verbose:
        print("\n================= SUMMARY =================")
        print(f"tasks scored        : {len(rows)}")
        print(f"missing (no onnx)   : {missing}  (these earn 0 on the real LB)")
        print(f"INCORRECT (fail>0)  : {n_incorrect}  <-- earning 0, highest priority")
        print(f"other problems      : {n_problem}")
        print(f"sum of points (this set): {total:.2f}")
        print(f"implied LB if missing=0 : {total:.2f} / 10000")
        print("\n----- BOTTOM 25 (attack these first) -----")
        print(f"{'task':>4} {'pts':>7} {'cost':>12} {'params':>9} {'mem':>9}  status")
        for r in rows[:25]:
            pts = f"{r['points']:.3f}" if r['points'] is not None else "n/a"
            print(f"{str(r['task']):>4} {pts:>7} "
                  f"{str(r['cost']):>12} {str(r['params']):>9} {str(r['memory']):>9}  {r['status']}")
        print("===========================================")
    return rows


## Cell 2 — Paths for the two submissions
For each submission, set EITHER the `_DIR` (a folder containing task*.onnx) OR the
`_ZIP` (a submission.zip). If you set `_ZIP`, leave `_DIR=None` and the notebook
unzips it. **Label A = your higher source (6154), B = the other (6126)** — labels
are only used for tie-breaks, scoring is symmetric otherwise.

In [ ]:
import os, glob, json, math, shutil, zipfile
import numpy as np, onnx

TASK_DIR = "/kaggle/input/competitions/neurogolf-2026"
WORK     = "/kaggle/working"

# ---- Submission A (e.g. 6154.71) ----
A_DIR = None   # current best (DIR)
A_ZIP = "/kaggle/input/notebooks/massimilianoghiotto/convolution-series-part-4/submission.zip"
# ---- Submission B (e.g. 6126.15) ----
B_DIR = "/kaggle/input/datasets/biohack44/neurogolf-6211-bundle"
B_ZIP = None   # 6151 FP16 bundle (ZIP) <-- set exact path

def _resolve(dir_, zip_, tag):
    if dir_:
        n=len(glob.glob(os.path.join(dir_,'task*.onnx')))
        print(f"[{tag}] using dir {dir_}  ({n} onnx)"); return dir_
    dst=f"{WORK}/_src_{tag}"; os.makedirs(dst,exist_ok=True)
    with zipfile.ZipFile(zip_) as z: z.extractall(dst)
    # find the dir that actually holds task*.onnx
    for root,_,files in os.walk(dst):
        if any(f.startswith("task") and f.endswith(".onnx") for f in files):
            n=len(glob.glob(os.path.join(root,'task*.onnx')))
            print(f"[{tag}] unzipped {zip_} -> {root}  ({n} onnx)"); return root
    raise RuntimeError(f"[{tag}] no task*.onnx found in {zip_}")

A = _resolve(A_DIR, A_ZIP, "A")
B = _resolve(B_DIR, B_ZIP, "B")
print("scoring source:", SOURCE)
print("task jsons:", len(glob.glob(os.path.join(TASK_DIR,'task*.json'))))
def load_task(t):
    p=f"{TASK_DIR}/task{t:03d}.json"
    return json.load(open(p)) if os.path.isfile(p) else None
def pa(t): return os.path.join(A,f"task{t:03d}.onnx")
def pb(t): return os.path.join(B,f"task{t:03d}.onnx")


## Cell 3 — Score both sources per task, pick the winner
For each task we score A and B through the official scorer (correctness + cost)
and select per the tie-break rules. Builds the blended bundle in `_stage`.

In [ ]:
stage=f"{WORK}/_stage"
if os.path.exists(stage): shutil.rmtree(stage)
os.makedirs(stage)

def score(path, ex):
    if not os.path.isfile(path): return None
    return audit_one(path, ex, run_correctness=(ex is not None))

def better(ra, rb):
    # returns 'A' or 'B'
    aok = ra is not None and ra["n_fail"]==0 and ra["status"]=="ok" and ra["cost"] is not None
    bok = rb is not None and rb["n_fail"]==0 and rb["status"]=="ok" and rb["cost"] is not None
    if aok and not bok: return 'A'
    if bok and not aok: return 'B'
    if aok and bok:
        if ra["cost"] < rb["cost"]: return 'A'
        if rb["cost"] < ra["cost"]: return 'B'
        return 'A'   # equal cost -> A
    # neither fully ok: choose fewer fails, then A
    af = ra["n_fail"] if ra else 10**9
    bf = rb["n_fail"] if rb else 10**9
    if af<=bf: return 'A'
    return 'B'

picks={'A':0,'B':0}; rows=[]; gain_vs_A=0.0; gain_vs_B=0.0
missingA=missingB=0
for t in range(1,401):
    ex=load_task(t)
    ra=score(pa(t),ex); rb=score(pb(t),ex)
    if ra is None: missingA+=1
    if rb is None: missingB+=1
    if ra is None and rb is None:
        print(f"task{t:03d}: MISSING in BOTH -> task will score 0"); continue
    w=better(ra,rb); picks[w]+=1
    src = pa(t) if w=='A' else pb(t)
    shutil.copy(src, os.path.join(stage,f"task{t:03d}.onnx"))
    pa_pts = ra["points"] if (ra and ra["points"] is not None) else 0.0
    pb_pts = rb["points"] if (rb and rb["points"] is not None) else 0.0
    win_pts = pa_pts if w=='A' else pb_pts
    gain_vs_A += win_pts - pa_pts
    gain_vs_B += win_pts - pb_pts
    rows.append((t,w,pa_pts,pb_pts,win_pts))

print(f"\npicks: A={picks['A']}  B={picks['B']}  (missingA={missingA}, missingB={missingB})")
print(f"staged tasks: {len(glob.glob(stage+'/task*.onnx'))}")
print(f"local gain vs A-only: +{gain_vs_A:.2f}   vs B-only: +{gain_vs_B:.2f}")


## Cell 4 — Show the tasks where the pick differs (who won what)

In [ ]:
diff=[r for r in rows if abs(r[2]-r[3])>1e-9]
diff.sort(key=lambda r: -abs(r[2]-r[3]))
print(f"{len(diff)} tasks where A and B differ in points. Top 30 by gap:")
print(f"{'task':>4} {'win':>3} {'A_pts':>7} {'B_pts':>7} {'used':>7}")
for t,w,ap,bp,wp in diff[:30]:
    print(f"{t:>4} {w:>3} {ap:>7.3f} {bp:>7.3f} {wp:>7.3f}")


## Cell 5 — Package blended submission.zip + final audit

In [ ]:
out=f"{WORK}/submission"
if os.path.exists(out+'.zip'): os.remove(out+'.zip')
shutil.make_archive(out,"zip",stage)
print("wrote",out+".zip  (",len(glob.glob(stage+'/task*.onnx')),"tasks)")
final=audit_dir(stage, task_dir=TASK_DIR, out_csv=f"{WORK}/blend_audit.csv", run_correctness=True)
tot=sum(r['points'] for r in final if r['points'] is not None)
print(f"\nblended local total (public-only): {tot:.2f}")
print("Selection blend: every task == one of your real submissions, so private-set")
print("behavior is preserved. Blended LB >= max(A_LB, B_LB).")
